> **Full pipeline:** For cohort generation, NODE loading, and training, see [`docs/new_drug_tutorial.ipynb`](../docs/new_drug_tutorial.ipynb). This notebook only generates ODE class code from a config.


#Cell 1: Define the Model Configuration (The "Lego" Blocks)


## 1. MODEL CONFIGURATION
### Users will change this dictionary to define the structure of their model.

In [6]:
model_config = {
    'absorption': 'first_order',   # options: 'iv', 'first_order'
    'compartments': 1,             # options: 1, 2, 3
    'elimination': 'linear',       # options: 'linear', 'michaelis_menten'
}

# 2. POPULATION PARAMETERS & VARIABILITY
population_params = {
    'ka': 1.2,
    'CL': 10.5, 'Vc': 45.0,
    'Q': 15.0,  'Vp': 60.0
}

ipv_omega = {
    'ka': 0.4, 'CL': 0.3, 'Vc': 0.2, 'Q': 0.3, 'Vp': 0.3
}

# 3. COVARIATES (Mapping user inputs to parameters)
covariates = {
    'CL': [
        {'name': 'WT', 'type': 'continuous', 'ref': 70.0, 'rel': 'power', 'theta_val': 0.75}
    ]
}

Cell 2: The Logic Engine (String Builders)
This cell contains the logic that reads the config and generates the Python syntax for the ODEs and parameter calculations.
Python

In [7]:
def build_param_extraction(config, covariates):
    """Builds the string to extract and scale parameters based on covariates."""
    lines = [
        "        # Extract base individual parameters",
        "        ka = self.individual_params.get('ka', 0.0)",
        "        CL = self.individual_params.get('CL', 0.0)",
        "        Vc = self.individual_params.get('Vc', 0.0)",
        "        Q = self.individual_params.get('Q', 0.0)",
        "        Vp = self.individual_params.get('Vp', 0.0)",
        "        Vmax = self.individual_params.get('Vmax', 0.0)",
        "        Km = self.individual_params.get('Km', 0.0)",
        "",
        "        # Apply Covariates (Assumes self.cov_data exists in the class)"
    ]
    
    for param, cov_list in covariates.items():
        for cov in cov_list:
            if cov['type'] == 'continuous' and cov['rel'] == 'power':
                lines.append(f"        {param} = {param} * ((self.cov_data['{cov['name']}'] / {cov['ref']}) ** {cov['theta_val']})")
    
    return "\n".join(lines)


def build_ode_system(config):
    """Builds the actual Differential Equations based on the selected architecture."""
    lines = ["        # ODE System Formulation"]
    
    # 1. State Unpacking
    state_vars = ["A_central"]
    if config['absorption'] == 'first_order':
        state_vars.insert(0, "A_gut")
    if config['compartments'] >= 2:
        state_vars.append("A_peri1")
    if config['compartments'] == 3:
        state_vars.append("A_peri2")
        
    lines.append(f"        {', '.join(state_vars)} = state")
    lines.append(f"        C_central = A_central / Vc")
    lines.append("")

    # 2. Absorption Logic
    if config['absorption'] == 'first_order':
        lines.append("        dA_gut_dt = -ka * A_gut")
        input_term = "ka * A_gut"
    else: # IV
        input_term = "0.0" # Dose goes directly to central state initially

    # 3. Elimination Logic
    if config['elimination'] == 'linear':
        elim_term = "(CL / Vc) * A_central"
    elif config['elimination'] == 'michaelis_menten':
        elim_term = "(Vmax * C_central) / (Km + C_central)"
    
    # 4. Distribution Logic
    dist_central = ""
    if config['compartments'] >= 2:
        lines.append("        dA_peri1_dt = (Q / Vc) * A_central - (Q / Vp) * A_peri1")
        dist_central += " - (Q / Vc) * A_central + (Q / Vp) * A_peri1"
    if config['compartments'] == 3:
        # Assuming Q2 and Vp2 exist in parameters for a 3-cpt model
        lines.append("        dA_peri2_dt = (Q2 / Vc) * A_central - (Q2 / Vp2) * A_peri2")
        dist_central += " - (Q2 / Vc) * A_central + (Q2 / Vp2) * A_peri2"

    # 5. Assemble Central Compartment
    lines.append(f"        dA_central_dt = {input_term} - {elim_term}{dist_central}")
    
    # 6. Return Statement
    lines.append("")
    return_vars = ["dA_gut_dt"] if config['absorption'] == 'first_order' else []
    return_vars.append("dA_central_dt")
    if config['compartments'] >= 2: return_vars.append("dA_peri1_dt")
    if config['compartments'] == 3: return_vars.append("dA_peri2_dt")
    
    lines.append(f"        return {', '.join(return_vars)}")
    
    return "\n".join(lines), len(state_vars)
# Generate the strings
param_string = build_param_extraction(model_config, covariates)
ode_string, num_states = build_ode_system(model_config)
initial_state_str = ", ".join(["0.0"] * num_states)


In [8]:
import textwrap

script_template = f"""import torch
import torch.nn as nn
from torchdiffeq import odeint_adjoint, odeint
import numpy as np

class GeneratedStandardPK(nn.Module):
    def __init__(self, cov_data, device=torch.device("cpu"), adjoint=False):
        super().__init__()
        self.device = device
        self.odeint = odeint_adjoint if adjoint else odeint
        self.cov_data = cov_data  # Dictionary of patient covariates (e.g., {{'WT': 80.0}})
        
        # Population parameters
        self.pop_params = {population_params}
        self.ipv = {ipv_omega}
        self.individual_params = {{}}

    def _sample_individual_parameters(self):
        for p_name, tv_p in self.pop_params.items():
            eta = torch.randn(1).item() * self.ipv.get(p_name, 0.0)
            self.individual_params[p_name] = torch.tensor(tv_p, device=self.device) * torch.exp(torch.tensor(eta, device=self.device))
        return self.individual_params

    def forward(self, t, state):
{param_string}

{ode_string}
    
    def get_initial_state(self):
        t0 = torch.tensor([0.0], device=self.device)
        state = tuple(torch.tensor([float(s)], device=self.device) for s in "{initial_state_str}".split(','))
        return t0, state
"""

file_name = "generated_standard_pk.py"
with open(file_name, "w", encoding="utf-8") as f:
    f.write(script_template)

print(f"✅ Generated Standard PK Script saved as '{file_name}'!")
print("-" * 50)
print(script_template) # Preview the generated code


✅ Generated Standard PK Script saved as 'generated_standard_pk.py'!
--------------------------------------------------
import torch
import torch.nn as nn
from torchdiffeq import odeint_adjoint, odeint
import numpy as np

class GeneratedStandardPK(nn.Module):
    def __init__(self, cov_data, device=torch.device("cpu"), adjoint=False):
        super().__init__()
        self.device = device
        self.odeint = odeint_adjoint if adjoint else odeint
        self.cov_data = cov_data  # Dictionary of patient covariates (e.g., {'WT': 80.0})

        # Population parameters
        self.pop_params = {'ka': 1.2, 'CL': 10.5, 'Vc': 45.0, 'Q': 15.0, 'Vp': 60.0}
        self.ipv = {'ka': 0.4, 'CL': 0.3, 'Vc': 0.2, 'Q': 0.3, 'Vp': 0.3}
        self.individual_params = {}

    def _sample_individual_parameters(self):
        for p_name, tv_p in self.pop_params.items():
            eta = torch.randn(1).item() * self.ipv.get(p_name, 0.0)
            self.individual_params[p_name] = torch.tensor(tv_p,